# Capability Lab — Distribution Compliance Gate v1
Launcher público seguro para uma futura release FINAL. Nenhum código privado ou credencial é armazenado aqui.
Pré-requisito: a infraestrutura de distribution compliance deve estar disponível na `main` privada.


In [ ]:
from google.colab import userdata
from getpass import getpass
import json, os, pathlib, subprocess

REPO = 'lucas-mateus-hq/lucas-capability-os'
REF = 'main'
WORK = pathlib.Path('/content/caplab-distribution')
PAGES = pathlib.Path('/tmp/caplab-pages')

def token_value():
    for key in ('GITHUB_TOKEN', 'CAPLAB_GITHUB_TOKEN', 'GH_TOKEN'):
        try:
            value = userdata.get(key)
            if value:
                return value
        except Exception:
            pass
    return getpass('GitHub token (não será exibido): ').strip()

token = token_value()
if not token:
    raise SystemExit('TOKEN_MISSING')
askpass = pathlib.Path('/content/caplab_distribution_askpass.sh')
askpass.write_text('#!/bin/sh\ncase \"$1\" in *Username*) echo \"x-access-token\";; *) printf \"%s\\n\" \"$CAPLAB_GH_TOKEN\";; esac\n', encoding='utf-8')
askpass.chmod(0o700)
env = os.environ.copy()
env['GIT_ASKPASS'] = str(askpass)
env['GIT_TERMINAL_PROMPT'] = '0'
env['CAPLAB_GH_TOKEN'] = token
try:
    subprocess.run(['rm', '-rf', str(WORK), str(PAGES)], check=True)
    subprocess.run(['git', 'clone', '--branch', REF, '--single-branch', f'https://github.com/{REPO}.git', str(WORK)], check=True, env=env)
    profile_path = WORK / 'docs/ip/distribution/DISTRIBUTION_PROFILE.json'
    if not profile_path.exists():
        raise SystemExit('DISTRIBUTION_INFRASTRUCTURE_NOT_ON_MAIN')
    profile = json.loads(profile_path.read_text(encoding='utf-8'))
    if profile.get('status') != 'FINAL':
        print(f"CAPLAB_DISTRIBUTION=BLOCKED status={profile.get('status')}")
    else:
        subprocess.run(['bash', '-lc', 'npm ci --ignore-scripts --no-audit --no-fund'], cwd=WORK, check=True)
        subprocess.run(['bash', '-lc', 'npm ci --ignore-scripts --no-audit --no-fund'], cwd=WORK/'artifacts/CS16_HOME_BASE_UI/prototype', check=True)
        subprocess.run(['python', '-m', 'pip', 'install', '-q', '-r', 'docs/ip/sbom/private-python.lock.txt'], cwd=WORK, check=True)
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/lucas-mateus-hq/lucas-capability-os-pages.git', str(PAGES)], check=True)
        subprocess.run(['bash', '-lc', f'npm ci --prefix {PAGES} --ignore-scripts --no-audit --no-fund'], check=True)
        subprocess.run(['python', 'scripts/ip/distribution_compliance_gate.py', '--collect'], cwd=WORK, check=True, env=env)
        print('CAPLAB_DISTRIBUTION=GREEN')
finally:
    env.pop('CAPLAB_GH_TOKEN', None)
    askpass.unlink(missing_ok=True)
    token = None

# O que isso faz: autentica só em runtime, reconstrói ambientes resolvidos e executa o gate privado apenas quando o perfil estiver FINAL.
